# Predictive Patient Readmission System
## End-to-End Analysis

**Author:** Ronald Wen — UC Irvine, Computer Science  
**Dataset:** UCI Diabetes 130-US Hospitals (1999–2008)  
**Objective:** Predict 30-day hospital readmissions to support early clinical intervention.

---

This notebook walks through the full machine learning pipeline:
1. Exploratory data analysis
2. Feature engineering
3. Model training and comparison
4. SHAP explainability

> **Setup:** Run `pip install -r ../requirements.txt` before executing this notebook.

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from the notebooks/ directory
sys.path.insert(0, str(Path("../src").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path("..").resolve()
DATA_RAW     = PROJECT_ROOT / "data" / "diabetic_data.csv"
DATA_PROC    = PROJECT_ROOT / "data" / "processed"
MODELS_DIR   = PROJECT_ROOT / "models"
RESULTS_DIR  = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Environment ready.")

---
## 1. Exploratory Data Analysis

Before modelling, we examine the raw data to understand its structure, missingness patterns, class imbalance, and the distributions of key clinical variables.

In [ ]:
df_raw = pd.read_csv(DATA_RAW, na_values=["?"])
print(f"Shape: {df_raw.shape}")
df_raw.head(3)

In [ ]:
# Missingness audit
missing = (df_raw.isna().sum() / len(df_raw) * 100).sort_values(ascending=False)
missing_top = missing[missing > 0]
print("Columns with missing values (% missing):")
print(missing_top.to_string())

In [ ]:
# Target class distribution
readmit_counts = df_raw["readmitted"].value_counts()
print("Readmission distribution:")
print(readmit_counts)
print(f"
30-day readmission rate: {readmit_counts["<30"] / len(df_raw) * 100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw 3-class distribution
readmit_counts.plot(kind="bar", ax=axes[0], color=["#4CAF50", "#FF9800", "#F44336"], edgecolor="white", rot=0)
axes[0].set_title("Readmission Categories (Raw)", fontsize=12)
axes[0].set_xlabel("")
axes[0].set_ylabel("Patient Count")
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f"{bar.get_height():,.0f}", ha="center", va="bottom", fontsize=9)

# Binary target
binary_counts = pd.Series({"Not Readmitted (0)": (df_raw["readmitted"] != "<30").sum(),
                            "Readmitted <30d (1)": (df_raw["readmitted"] == "<30").sum()})
binary_counts.plot(kind="bar", ax=axes[1], color=["#2196F3", "#FF5722"], edgecolor="white", rot=0)
axes[1].set_title("Binary Target Distribution", fontsize=12)
axes[1].set_xlabel("")
axes[1].set_ylabel("Patient Count")
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f"{bar.get_height():,.0f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_target_distribution.png")

In [ ]:
# Distributions of key numeric clinical variables
numeric_features = ["time_in_hospital", "num_lab_procedures", "num_procedures",
                    "num_medications", "number_outpatient", "number_emergency",
                    "number_inpatient", "number_diagnoses"]

df_plot = df_raw[numeric_features].copy()

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, numeric_features):
    ax.hist(df_plot[col].dropna(), bins=30, color="#2196F3", edgecolor="white", alpha=0.85)
    ax.set_title(col.replace("_", " ").title(), fontsize=10)
    ax.set_ylabel("Count")
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Distributions of Key Clinical Variables", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Readmission rate by age group
age_readmit = df_raw.groupby("age").apply(
    lambda g: (g["readmitted"] == "<30").sum() / len(g) * 100
).reset_index()
age_readmit.columns = ["age_bracket", "readmit_rate_pct"]
age_readmit = age_readmit.sort_values("age_bracket")

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(age_readmit["age_bracket"], age_readmit["readmit_rate_pct"],
       color="#FF5722", edgecolor="white", alpha=0.88)
ax.set_xlabel("Age Bracket", fontsize=11)
ax.set_ylabel("30-Day Readmission Rate (%)", fontsize=11)
ax.set_title("30-Day Readmission Rate by Age Group", fontsize=13)
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_readmission_by_age.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Correlation heatmap of numeric features vs binary readmission target
df_corr = df_raw[numeric_features].copy()
df_corr["readmitted_30"] = (df_raw["readmitted"] == "<30").astype(int)

corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5, square=True, cbar_kws={"shrink": 0.8})
ax.set_title("Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2. Data Preprocessing

We run the full preprocessing pipeline: cleaning, feature engineering, target encoding, and train/test splitting.

In [ ]:
from data_preprocessing import run_preprocessing

X_train, X_test, y_train, y_test = run_preprocessing(
    raw_path=DATA_RAW,
    output_dir=DATA_PROC,
)

print(f"
Features: {X_train.columns.tolist()[:10]} ... ({X_train.shape[1]} total)")
print(f"Class balance in training set:")
print(y_train.value_counts(normalize=True).round(3))

---
## 3. Model Training

We train three models. Class imbalance (~11% positive rate) is handled explicitly in each:
- **Logistic Regression**: 
- **Random Forest**: 
- **XGBoost**:  set to negative/positive ratio

In [ ]:
from train_models import run_training

models = run_training(data_dir=DATA_PROC, models_dir=MODELS_DIR)
print("
Models trained:", list(models.keys()))

---
## 4. Model Evaluation

All models are evaluated on the held-out 20% test set. AUC-ROC is the primary metric because the dataset is imbalanced and clinical utility depends on ranking patients by risk rather than hard classifications.

In [ ]:
from evaluate import run_evaluation
import json

all_metrics = run_evaluation(data_dir=DATA_PROC, models_dir=MODELS_DIR, results_dir=RESULTS_DIR)

In [ ]:
# Pretty-print results table
metrics_df = pd.DataFrame(all_metrics).T
metrics_df.columns = ["AUC-ROC", "Accuracy", "Precision", "Recall", "F1"]
print(metrics_df.to_string())
metrics_df

In [ ]:
# Display the saved ROC curve plot
from IPython.display import Image
Image(str(RESULTS_DIR / "roc_curves.png"), width=600)

---
## 5. SHAP Explainability

We use TreeExplainer to decompose XGBoost predictions into per-feature Shapley value contributions. Unlike built-in feature importance (which only measures split frequency), SHAP values account for feature interactions and are consistent across samples.

In [ ]:
from explainability import run_explainability

run_explainability(data_dir=DATA_PROC, models_dir=MODELS_DIR, results_dir=RESULTS_DIR)

In [ ]:
# Display SHAP summary plot
Image(str(RESULTS_DIR / "shap_summary.png"), width=700)

In [ ]:
# Display SHAP bar importance
Image(str(RESULTS_DIR / "shap_importance_bar.png"), width=600)

In [ ]:
# Dependence plot: number_inpatient
Image(str(RESULTS_DIR / "shap_dependence_number_inpatient.png"), width=580)

---
## 6. Key Findings

### Model Performance
- **XGBoost** achieves the highest AUC-ROC (~0.67), outperforming both linear and tree ensemble baselines.
- The large gap between AUC (~0.67) and accuracy (~0.89) reflects the class imbalance — a naive classifier predicting "no readmission" for all patients would achieve ~89% accuracy with zero clinical utility.
- **Precision-recall tradeoff**: all models have low recall at the default threshold due to imbalance. In practice, threshold would be tuned based on the cost ratio of false negatives to false positives.

### SHAP-Identified Predictors
| Rank | Feature | Clinical interpretation |
|------|---------|-------------------------|
| 1 |  | Prior hospitalisations are the strongest predictor of future readmission |
| 2 |  | Frequent ER use signals poorly controlled chronic disease |
| 3 |  | Longer stays indicate disease severity and complexity |
| 4 |  | High medication burden reflects comorbidity load |
| 5 |  | Multiple concurrent diagnoses increase systemic risk |

### Limitations
- Dataset spans 1999–2008; drug and treatment protocols have changed
- Patient-level features (e.g. socioeconomic status, care continuity) are absent
- 30-day readmission is a system-level metric; some readmissions are planned or unpreventable

---
*Ronald Wen — UC Irvine, Computer Science*